# withGyro Experiment 0.2 — nonlinear temporal decoder probe

Analysis-only notebook for the finalized `structured_residual_gru_v1` artifacts. The training pipeline compares `accel30`, `angular30`, and `combined60` under `fixed250` and `relative10` with Linear, Linear+Local, Linear+Transition, Linear+Local+Transition, and GRU decoders. No SNN is used.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd

repo_root = Path.cwd().resolve().parent.parent if Path.cwd().name == 'withGyro' else Path.cwd().resolve()
while not (repo_root / 'AGENTS.md').is_file():
    if repo_root.parent == repo_root:
        raise FileNotFoundError('Could not locate repository root')
    repo_root = repo_root.parent
artifact_root = (
    repo_root / 'notebooks' / 'artifacts' / 'withGyro' /
    'experiment_0_2_nonlinear_temporal_decoder_probe' / 'structured_residual_gru_v1'
)
required = {
    'results': artifact_root / 'experiment_0_2_results.csv',
    'summary': artifact_root / 'experiment_0_2_summary.csv',
    'paired': artifact_root / 'experiment_0_2_paired_decoder_deltas.csv',
    'sensor': artifact_root / 'experiment_0_2_sensor_deltas.csv',
    'synergy': artifact_root / 'experiment_0_2_nonlinear_synergy.csv',
    'conclusion': artifact_root / 'experiment_0_2_conclusion.json',
}
missing = [str(path) for path in required.values() if not path.is_file()]
if missing:
    raise FileNotFoundError('Run/finalize Experiment 0.2 first. Missing:\n' + '\n'.join(missing))
results = pd.read_csv(required['results'])
summary = pd.read_csv(required['summary'])
paired = pd.read_csv(required['paired'])
sensor = pd.read_csv(required['sensor'])
synergy = pd.read_csv(required['synergy'])
conclusion = json.loads(required['conclusion'].read_text(encoding='utf-8'))
conclusion

## Absolute test Balanced Accuracy

The main table is paired across the five user-disjoint split seeds.

In [ ]:
display(
    summary.pivot_table(
        index=['representation', 'decoder'],
        columns='channel_set',
        values='mean_test_ba',
    ).round(4)
)

In [ ]:
decoder_order = ['linear', 'local', 'transition', 'local_transition', 'gru']
for representation in ['fixed250', 'relative10']:
    view = summary[summary.representation == representation].copy()
    fig, ax = plt.subplots(figsize=(9, 5))
    pivot = view.pivot(index='decoder', columns='channel_set', values='mean_test_ba').reindex(decoder_order)
    pivot.plot(kind='bar', ax=ax)
    ax.set_title(f'{representation}: mean test Balanced Accuracy')
    ax.set_ylabel('Balanced Accuracy')
    ax.set_xlabel('Decoder')
    ax.tick_params(axis='x', rotation=20)
    ax.legend(title='Channel set')
    fig.tight_layout()
    plt.show()

## Structured nonlinear gain over frozen Linear

Positive values indicate information recovered beyond the position-aware Linear baseline.

In [ ]:
gain_columns = [
    'local_minus_linear',
    'transition_minus_linear',
    'local_transition_minus_linear',
    'gru_minus_linear',
]
gain_summary = (
    paired.groupby(['representation', 'channel_set'])[gain_columns]
    .agg(['mean', 'std'])
)
display(gain_summary.round(4))

## Nonlinear sensor-fusion synergy

`combined_gain_minus_angular_gain > 0` means the residual improvement above Linear is larger for `combined60` than for `angular30`. This is evidence for additional nonlinear complementarity, not by itself proof of a specific cross-modal interaction mechanism.

In [ ]:
display(
    synergy.groupby(['representation', 'decoder'])[[
        'combined_gain_minus_angular_gain',
        'combined_gain_minus_accel_gain',
    ]].agg(['mean', 'std']).round(4)
)

## Structured residual versus GRU

In [ ]:
display(
    paired.groupby(['representation', 'channel_set'])['local_transition_minus_gru']
    .agg(['mean', 'std', lambda s: int((s > 0).sum())])
    .rename(columns={'<lambda_0>': 'local_transition_better_splits'})
    .round(4)
)